In [ ]:
# 04 Experiment B: ranking-aware regression (declared grid)
#
# Question. Does adding a pairwise ranking term to MSE change the ordering quality (PLCC / SROCC / KROCC) relative to A without materially worsening absolute error (MSE / MAE / RMSE)?
# Hypothesis H-B (to be tested, not assumed). For some lambda > 0, rank correlation rises while validation RMSE stays within 5% of A. The null (no detectable change) is equally plausible.
# Difference from A. Only the loss: L_total = L_MSE + lambda * L_rank. Model, data, preprocessing, optimizer, epochs, batch size, seed and selection rule are identical (section 1 shows this).
# Procedure. Stage 1 runs the declared lambda grid, the declared rule selects one point using validation data only, stage 2 runs the declared one-at-a-time sensitivity. Spec: docs/experiments/b-ranking-aware.md section 7.
# Cell labels. [SAFE] inspection only | [TRAINS] trains, OFF by default | [READS RUN] reads completed runs | [WRITES] writes a selection record, OFF by default. No cell reads the official test set or the calibration partition.
from pathlib import Path
import os

REPO = Path.cwd()
while not (REPO / "pyproject.toml").exists() and REPO != REPO.parent:
    REPO = REPO.parent
os.chdir(REPO)

import json
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from ct_iqa.config.loader import load_study_config
from ct_iqa.data.splits import load_manifest
from ct_iqa.experiment import report
from ct_iqa.experiment.grid import apply_overrides
from ct_iqa.experiment.run import run_experiment
from ct_iqa.experiment.runs import read_predictions, run_summary, runs_table, select_run
from ct_iqa.models.weights import sha256_file

from ct_iqa.data.sampling import ordered_pair_mask
from ct_iqa.experiment import grid
from ct_iqa.losses.mse import MSELoss
from ct_iqa.losses.ranking import MSERankingLoss
from ct_iqa.models.registry import ModelOutput


def show(title, obj):
    """Print a section title, then the table as plain text."""
    print(f"\n=== {title} ===")
    print(obj.to_string())   # plain text table, no interactive viewer



In [ ]:
# 1. Configuration: A and B side by side [SAFE]
# Rows that differ are listed first. The YAML values of loss.ranking are placeholders; every grid run overrides them with a declared grid point, and the override is recorded in its run_manifest.json.
CONFIG = "configs/experiments/b_ranking_aware.yaml"
cfg = load_study_config(CONFIG)        # strict: unknown keys or invalid values raise here, before anything runs
show(f"Configuration: configs/base.yaml + {CONFIG}", report.config_frame(**{cfg.run.experiment_id: cfg.to_dict()}))

cfg_a = load_study_config("configs/experiments/a_reference.yaml")
side = report.config_frame(A=cfg_a.to_dict(), B=cfg.to_dict())
differs = side["A"].astype(str) != side["B"].astype(str)
show("Settings that differ between A and B", side[differs])
print("All other", int((~differs).sum()), "settings are identical.")


In [ ]:
# 2. The objective [SAFE]
# Per mini-batch with predicted means f_i (normalized 0-1) and raw labels y_i (0-4):
# P = {(i, j) : y_i - y_j >= delta}            ordered pairs, i is the higher-quality image; ties, pairs closer than delta and i = j are excluded
# L_rank = mean over P of max(0, margin - (f_i - f_j))   hinge: zero once i is ranked above j by at least margin
# L_total = L_MSE + lambda * L_rank,  L_MSE = mean (f_i - y_i / 4)^2
# delta (min_score_diff) is in raw 0-4 units, margin in the normalized 0-1 space. If P is empty, L_rank = 0 with zero gradient. lambda = 0 gives exactly A's objective. Code: ct_iqa/losses/ranking.py, ct_iqa/data/sampling.py.
# 3. Label grid and delta (training labels only) [SAFE]
# LDCT-IQAC scores lie on a discrete grid, so delta acts as the smallest grid step at or above it. A delta lying exactly on a grid point would let float32 rounding flip pairs; the declared deltas are checked to lie between grid points. Uses the 800 training labels only.
labels = json.loads(Path(cfg.data.train_json_path).read_text())
manifest = load_manifest(cfg.data.split_manifest, labels)      # raises if the frozen manifest was altered
y_train = np.array([labels[n] for n in manifest["train"]])
levels = np.unique(y_train)
step = float(np.round(np.diff(levels).min(), 6))
show(f"Training label grid: {len(levels)} levels, step {step}", pd.Series(y_train).value_counts().sort_index().rename("images").to_frame().T)

n = len(y_train)
diff = np.subtract.outer(y_train, y_train)
rows = []
for delta in (0.0, grid.STAGE1_MIN_SCORE_DIFF, *grid.STAGE2_MIN_SCORE_DIFFS):
    eff = step * np.ceil(delta / step - 1e-9) if delta > 0 else step
    off_grid = delta == 0 or abs(delta / step - round(delta / step)) > 0.1
    rows.append({"delta": delta, "effective minimum difference": round(eff, 3), "delta off the grid": off_grid,
                 "eligible ordered pairs (all 800 train)": int((diff >= max(delta, 1e-12)).sum()),
                 "fraction of unordered pairs": round(float((diff >= max(delta, 1e-12)).sum()) / (n * (n - 1) / 2), 3)})
show("Pairs eligible for the ranking term, by delta (delta = 0 is a reference row: ties are never pairs)", pd.DataFrame(rows).set_index("delta"))
assert all(r["delta off the grid"] for r in rows), "a declared delta lies on a grid point: float32 rounding could flip pairs"

rng = np.random.default_rng(cfg.train.seed)
batch = torch.tensor(rng.choice(y_train, size=cfg.train.batch_size, replace=False), dtype=torch.float32)
pairs = {d: int(ordered_pair_mask(batch, d).sum()) for d in (grid.STAGE1_MIN_SCORE_DIFF, *grid.STAGE2_MIN_SCORE_DIFFS)}
print(f"One random batch of {cfg.train.batch_size} training labels:", pairs, f"of {cfg.train.batch_size * (cfg.train.batch_size - 1) // 2} possible pairs")


In [ ]:
# 4. Pair-mask and objective preflight [SAFE]
# Small hand-made tensors, no model and no data: pair direction, excluded ties, the threshold boundary, an empty eligible-pair batch, and lambda = 0 reproducing MSE exactly. Expected: every row True.
y = torch.tensor([1.0, 1.4, 1.0, 3.0])
M = ordered_pair_mask(y, 0.25)
f = torch.tensor([0.30, 0.20, 0.25, 0.80], requires_grad=True)
out = ModelOutput(mean=f, log_var=None)
rank_only = MSERankingLoss(1.0, 0.05, 0.25)(out, y / 4, y)
empty = MSERankingLoss(1.0, 0.05, 0.25)(out, torch.full((4,), 0.5), torch.full((4,), 2.0))
empty.total.backward()
checks = {
    "higher-scored image comes first (1.4 > 1.0: M[1, 0])": bool(M[1, 0]) and not bool(M[0, 1]),
    "ties excluded (1.0 vs 1.0)": not bool(M[0, 2]) and not bool(M[2, 0]),
    "diagonal excluded": not bool(M.diagonal().any()),
    "boundary: difference exactly delta is a pair": bool(ordered_pair_mask(torch.tensor([1.0, 1.25]), 0.25)[1, 0]),
    "empty pair set gives rank = 0 and a finite total": empty.parts["n_valid_pairs"] == 0 and empty.parts["rank"] == 0.0 and bool(torch.isfinite(empty.total)),
    "lambda = 0 equals MSE exactly": float(MSERankingLoss(0.0, 0.05, 0.25)(out, y / 4, y).total) == float(MSELoss()(out, y / 4, y).total),
    "mis-ordered pairs give a positive ranking loss": rank_only.parts["rank"] > 0,
}
show("Ranking objective preflight", pd.DataFrame({"passed": checks}))
assert all(checks.values())


In [ ]:
# 5. Preflight [SAFE]
# Checks inputs WITHOUT training and WITHOUT building or running a model: frozen split manifest (hash, disjoint partitions, coverage), pretrained weights SHA-256, image files present (counted, not read), CUDA availability, existing runs. Expected output: a table where every row is True.
labels = json.loads(Path(cfg.data.train_json_path).read_text())
manifest = load_manifest(cfg.data.split_manifest, labels)      # raises if the frozen manifest was altered
sizes = {p: len(manifest[p]) for p in ("train", "validation", "calibration")}
weights = Path(cfg.model.pretrained_weights)
n_images = len(list(Path(cfg.data.train_image_dir).glob("*.tif*")))
preflight = {
    "split manifest verified (hash, disjoint, complete)": True,
    "partitions are 800 / 100 / 100": list(sizes.values()) == [800, 100, 100],
    "pretrained weights file present": weights.is_file(),
    "pretrained weights SHA-256 matches config": weights.is_file() and sha256_file(weights) == cfg.model.pretrained_sha256,
    "1000 training images present (counted, not read)": n_images == 1000,
}
show("Preflight checks", pd.DataFrame({"passed": preflight}))
show("Environment", pd.DataFrame({"value": {
    "split manifest SHA-256": manifest["manifest_sha256"], "split seed": manifest["seed"],
    "torch": torch.__version__, "CUDA available": torch.cuda.is_available(),
    "CUDA device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-"}}))
print("PREFLIGHT PASSED" if all(preflight.values()) else "PREFLIGHT FAILED - fix the False rows before training")


In [ ]:
# 6. Stage 1: declared lambda grid [TRAINS - OFF by default]
# Points: lambda in {0.03, 0.1, 0.3, 1.0}, margin 0.05, delta 0.25, seed 42 (as declared). One full training run per point, each in its own run directory with the point recorded as an override. Points that already have a completed run are skipped, so re-running never duplicates finished work; a failed point stops the loop with its error and is listed as failed below. Expected time: 4 x one Experiment A run.
RUN_GRID_STAGE1 = False
DEVICE = "cuda"
GRID_SEED = cfg.train.seed

stage1 = grid.b_grid_stage1()
todo = grid.pending_points(stage1, GRID_SEED)
show(f"Stage 1 grid (seed {GRID_SEED})", pd.DataFrame(stage1).assign(pending=[p in todo for p in stage1]))
if RUN_GRID_STAGE1:
    if DEVICE.startswith("cuda") and not torch.cuda.is_available():
        raise RuntimeError(f"DEVICE={DEVICE!r} but CUDA is not available; select the GPU kernel or set DEVICE = 'cpu' explicitly")
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", datefmt="%H:%M:%S", force=True)
    for d in grid.run_grid(cfg, todo, [GRID_SEED], DEVICE, CONFIG):
        print("COMPLETED:", d.as_posix())
else:
    print("RUN_GRID_STAGE1 is False: nothing was trained.")


In [ ]:
# 7. Grid status [SAFE]
# All B runs: configuration, seed, run ID, status, selected epoch and validation metrics (selection-biased).
rows = runs_table(grid.B_EXPERIMENT_ID)
cols = ["status", "lambda_rank", "margin", "min_score_diff", "seed", "best_epoch", "plcc", "srocc", "krocc", "mse", "mae", "rmse", "error"]
show("Experiment B runs", pd.DataFrame(rows).set_index("run_id")[cols].round(4) if rows else pd.DataFrame({"status": {"(none)": "no B run yet: PENDING"}}))


In [ ]:
# 8. Stage-1 selection by the declared rule [READS RUN; WRITES only if enabled]
# Exploratory model selection on the VALIDATION partition only (never the calibration partition or the test set). Rule (declared): keep points whose validation RMSE is within 5% of A's; take the highest mean of PLCC, SROCC and KROCC; claim an improvement only if that mean exceeds A's by more than the paired-bootstrap 95% half-width (here: bootstrap of the mean-correlation difference over the 100 images; optimistic because patient grouping is unknown). The rule needs an explicit completed A run (A_RUN_ID, see notebook 03) and refuses to run on an incomplete or ambiguous stage. WRITE_SELECTION_RECORD = True saves the record (rule, inputs, candidates, reason) to experiments/study_v2/B_ranking_aware_resnet50/; it is never overwritten.
A_RUN_ID = None
WRITE_SELECTION_RECORD = False

record = None
if A_RUN_ID is None:
    print("Set A_RUN_ID to a completed Experiment A run (notebook 03) to apply the selection rule: PENDING.")
else:
    a_dir = select_run(cfg_a.run.experiment_id, A_RUN_ID)
    record = grid.select_stage1(a_dir, GRID_SEED)
    show("Stage-1 candidates (validation, selection-biased)", pd.DataFrame(record["candidates"]).set_index("run_id").round(4))
    print(f"Reference A run {record['reference_run']}: RMSE {record['reference_rmse']:.4f}, mean correlation {record['reference_mean_corr']:.4f}")
    if record.get("bootstrap_vs_A"):
        show("Selected point vs A: paired bootstrap of the mean-correlation difference", pd.DataFrame({"value": record["bootstrap_vs_A"]}).round(4))
    print("SELECTED:", record["selected"] and {k: record["selected"][k] for k in ("lambda_rank", "margin", "min_score_diff", "run_id")})
    print("OUTCOME:", record["outcome"])
    if WRITE_SELECTION_RECORD:
        print("selection record written:", grid.write_selection_record(record).as_posix())


In [ ]:
# 9. Stage 2: one-at-a-time sensitivity [TRAINS - OFF by default]
# At the stage-1 lambda only: margin in {0.02, 0.1} (delta 0.25) and delta 0.5 (margin 0.05). Set SELECTED_LAMBDA by hand from the stage-1 record; the cell refuses a lambda outside the declared grid. Stage-2 results are reported as a sensitivity table; how they enter the final choice is not specified by the declared rule.
RUN_GRID_STAGE2 = False
SELECTED_LAMBDA = None

if SELECTED_LAMBDA is None:
    print("SELECTED_LAMBDA is not set: stage 2 is PENDING.")
else:
    stage2 = grid.b_grid_stage2(SELECTED_LAMBDA)
    todo2 = grid.pending_points(stage2, GRID_SEED)
    show(f"Stage 2 points (lambda {SELECTED_LAMBDA}, seed {GRID_SEED})", pd.DataFrame(stage2).assign(pending=[p in todo2 for p in stage2]))
    if RUN_GRID_STAGE2:
        if DEVICE.startswith("cuda") and not torch.cuda.is_available():
            raise RuntimeError(f"DEVICE={DEVICE!r} but CUDA is not available")
        logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", datefmt="%H:%M:%S", force=True)
        for d in grid.run_grid(cfg, todo2, [GRID_SEED], DEVICE, CONFIG):
            print("COMPLETED:", d.as_posix())
    rows2 = [r for r in runs_table(grid.B_EXPERIMENT_ID) if r["status"] == "completed" and r["lambda_rank"] == SELECTED_LAMBDA and r["seed"] == GRID_SEED]
    if rows2:
        show("Sensitivity at the selected lambda (validation, selection-biased)", pd.DataFrame(rows2).set_index("run_id")[cols].round(4))


In [ ]:
# 10. Inspect one B run [READS RUN]
# The cells below inspect ONE run, chosen explicitly. With several completed grid runs RUN_ID must be set (normally the selected run_id from the stage-1 record); the notebook never takes the latest run.
# 11. Select and verify one completed run [READS RUN]
# Set RUN_ID to a run directory name from the table above. With exactly one completed run it may stay None; with several the cell stops rather than guess. Verification: status completed, every artifact matches its recorded SHA-256, history complete, same frozen split and bn_eps as the current config, official test set unused. All later cells are skipped while no run is selected.
RUN_ID = None

run_dir = select_run(cfg.run.experiment_id, RUN_ID)
if run_dir is None:
    print(f"No completed run of {cfg.run.experiment_id}: results are PENDING.")
else:
    S = run_summary(run_dir)                               # raises on failed / incomplete / altered runs
    m, history, metrics = S["manifest"], S["history"], S["metrics"]
    verified = {
        "status is completed": m["status"] == "completed",
        "artifact SHA-256 all match": True,                # run_summary raised otherwise
        "history marked complete": history.get("complete", True),
        "split manifest SHA-256 = current frozen manifest": m["split_manifest_sha256"] == manifest["manifest_sha256"],
        "bn_eps = current config": m["bn_eps"] == cfg.model.bn_eps,
        "official test set unused": m["official_test_set_used"] is False,
    }
    show(f"Verification of {run_dir.as_posix()}", pd.DataFrame({"passed": verified}))
    if not all(verified.values()):
        raise RuntimeError("this run does not match the current protocol; do not interpret it")


In [ ]:
# 12. Selected checkpoint [READS RUN]
# The checkpoint is the epoch with the highest validation PLCC (raw 0-4 scale), evaluated after every epoch; a later epoch replaces it only on a strict improvement. The validation partition therefore SELECTS the reported model: validation metrics below are optimistic (selection-biased).
if run_dir:
    best = next(e for e in history["epochs"] if e["epoch"] == history["best_epoch"])
    show("Checkpoint selection record", pd.DataFrame({"value": {
        "selection policy": history["selection_policy"], "selected epoch (1-based)": history["best_epoch"],
        "of epochs": len(history["epochs"]), "selected validation PLCC": round(history["best_value"], 4),
        "validation PLCC at the final epoch": round(history["epochs"][-1]["val_plcc"], 4),
        "checkpoint file": (run_dir / "checkpoints" / "best.pt").as_posix(),
        "checkpoint SHA-256": m["artifact_sha256"]["checkpoints/best.pt"]}}))


In [ ]:
# 13. Training curves [READS RUN]
# Left: training and validation objective per epoch (normalized 0-1 score space). Right: validation PLCC and SROCC. The dashed line marks the selected epoch.
if run_dir:
    report.plot_history(history, f"{m['run_id']}  (n_train = 800, n_val = 100)")
    plt.show()
    show("Per-epoch history (selected = True: new best checkpoint at that epoch)", report.history_frame(history).round(4))


In [ ]:
# 14. Objective components [READS RUN]
# Each term of the objective separately (train solid, validation dashed) and, for B, the fraction of in-batch pairs that were eligible.
if run_dir:
    report.plot_loss_parts(history, "Objective components per epoch")
    plt.show()
    show("Training-loss components per epoch (sample-weighted batch means)", report.loss_parts_frame(history).round(5))


In [ ]:
# 15. Validation metrics at the selected checkpoint [READS RUN]
# Raw 0-4 score scale, n = 100 validation images. SELECTION-BIASED (the same images chose the epoch); not comparable with historical official-test numbers, which come from a different partition and protocol.
if run_dir:
    show(f"Validation metrics, epoch {metrics['epoch_1based']}, n = {metrics['n']}, {metrics['scale']} (selection-biased)",
         report.metrics_frame(metrics))


In [ ]:
# 16. Prediction and residual diagnostics [READS RUN]
# Scatter against the identity line, residual distribution, residual vs label, and errors per label range with the number of images in each range (ranges with few images are noisy).
if run_dir:
    p = read_predictions(run_dir)
    y, f = np.array(p["label_0_4"]), np.array(p["prediction_0_4"])
    report.plot_prediction_scatter(y, f, f"Validation: prediction vs label (n = {len(y)})")
    plt.show()
    report.plot_residuals(y, f, "Validation residuals")
    plt.show()
    show("Error by label range (score units)", report.error_by_score_bin(y, f).round(3))


In [ ]:
# 17. Reproducibility record [READS RUN]
# Everything needed to identify and reproduce this run; all paths are relative to the repository root. The full configuration is in config.snapshot.json inside the run directory.
if run_dir:
    show("Reproducibility record", report.reproducibility_frame(S))

# 18. Limitations and next step
# Limitations. Grid selection uses the same 100 validation images that select checkpoints, so the selected point's validation numbers are doubly optimistic; one seed per point; in-batch pairs depend on batch composition; the image-level bootstrap ignores possible patient clustering.
# Next step. Keep the stage-1 record, then compare the selected run with A in notebook 06. Additional seeds for the selected configuration and the final evaluation partition are pending the protocol decision.
